# 🌙 Aula 18+19 — Concorrência e Energia na GPU

**Curso:** Técnico em Inteligência Artificial | **UC:** Introdução a Arquitetura de Computadores

**Objetivo:** operar a GPU com **justiça** (1 job por vez + fila com prioridade) e com
**eficência** (medir e limitar a energia). O notebook reúne os temas de **concorrência** e
**energia** num roteiro único.

**Roteiro:**
1. Diagnóstico do ambiente (GPU, `nvidia-smi`, `flock`, PyTorch).
2. **Parte 1 — Concorrência:** race condition, lock por diretório e fila por prioridade
   (com **processos concorrentes de verdade**).
3. **Parte 2 — Energia:** coleta **real** de temperatura/potência, throughput e eficiência
   (imgs/J), além do trade-off do Power Limit.
4. Integração, síntese e **Exercícios (5)**.

> 💡 **No Colab com GPU T4** a temperatura, a potência e o throughput são **medidos de
> verdade**. Sem GPU NVIDIA, o notebook entra em **modo simulado** e mostra valores de
> referência — os conceitos e o código continuam válidos.


## 1. Diagnóstico do ambiente

Descobrimos o que existe aqui: GPU NVIDIA? `nvidia-smi`/`pynvml`? `flock`? PyTorch? Isso define
se as medições serão **reais** ou **simuladas**.


In [ ]:
# @title 🔍 O que este ambiente oferece?
# ============================================================================
# OBJETIVO: detectar GPU NVIDIA, bibliotecas de leitura e PyTorch.
# ============================================================================
import importlib.util
import shutil

def tem(modulo):
    return importlib.util.find_spec(modulo) is not None

TEM_SMI   = shutil.which("nvidia-smi") is not None
TEM_FLOCK = shutil.which("flock") is not None
TEM_NVML  = tem("pynvml") or tem("nvidia_ml_py")
TEM_TORCH = tem("torch")

print(f"nvidia-smi : {'sim' if TEM_SMI else 'nao'}")
print(f"pynvml     : {'sim' if TEM_NVML else 'nao'}")
print(f"flock      : {'sim' if TEM_FLOCK else 'nao (usaremos lock por diretorio)'}")
print(f"torch      : {'sim' if TEM_TORCH else 'nao'}")

if TEM_TORCH:
    import torch
    print(f"CUDA       : {'sim' if torch.cuda.is_available() else 'nao (CPU)'}")
    if torch.cuda.is_available():
        print(f"GPU        : {torch.cuda.get_device_name(0)}")

MODO = "real" if (TEM_SMI or TEM_NVML) else "simulado"
print(f"\n>>> Medicoes de energia: {MODO.upper()}")


## Parte 1 — Concorrência

### 2. O problema: 4 jobs disputando 1 GPU

Sem controle, os jobs competem pela VRAM ao mesmo tempo e causam **CUDA OOM**. A GPU é um
recurso **exclusivo**: garantimos **um job por vez** com um **lock** e organizamos a ordem com
uma **fila**.


In [ ]:
# @title 💥 Por que sem lock da errado (ilustracao)
# ============================================================================
# OBJETIVO: ilustrar a corrida pela VRAM. A GPU tem 4 GB no exemplo e cada job
# pede 2.5 GB: sem coordenacao, os jobs se atropelam e estouram a memoria.
# ============================================================================
VRAM_TOTAL = 4.0
VRAM_USADA = 0.0

def alocar(nome, gb):
    global VRAM_USADA
    if VRAM_USADA + gb > VRAM_TOTAL:
        print(f"[{nome}] CUDA OOM! (pediu {gb} GB; restam {VRAM_TOTAL - VRAM_USADA:.1f} GB)")
        return False
    VRAM_USADA += gb
    print(f"[{nome}] alocou {gb} GB (total {VRAM_USADA:.1f}/{VRAM_TOTAL} GB)")
    return True

print("=== SEM LOCK: 4 jobs ao mesmo tempo ===")
for job in ["A", "B", "C", "D"]:
    alocar(f"Job-{job}", 2.5)
print("\nSem exclusao mutua, os jobs se destroem (OOM).")


### 3. A fila com lock (código real)

Agora criamos dois arquivos — um **job de treino** e a **fila com lock** — e lançamos **4
processos concorrentes de verdade**. O lock é um **diretório** (`os.mkdir`, atômico) e a fila
ordena os *tickets* por prioridade.


In [ ]:
%%writefile train_job.py
# train_job.py - job de "treino" simulado (sem exigir GPU)
# (celula @title: Criar train_job.py - job de treino simulado)
import argparse
import time

parser = argparse.ArgumentParser()
parser.add_argument("--nome", default="Job")
parser.add_argument("--epocas", type=int, default=3)
args = parser.parse_args()

for epoca in range(1, args.epocas + 1):
    print(f"[{args.nome}] epoca {epoca}/{args.epocas} (loss={1.0/epoca:.3f})")
    time.sleep(0.4)  # consome tempo, como um treino real
print(f"[{args.nome}] concluido")


In [ ]:
%%writefile fila_gpu.py
# fila_gpu.py - fila por prioridade com exclusao mutua (1 job por vez)
# Uso: python fila_gpu.py <prioridade> <nome> <script> [args...]
import os
import subprocess
import sys
import time

SPOOL = "gpu_spool"
LOCK = "gpu.lockdir"
LOG = "fila.log"

def log(msg):
    linha = f"[{time.strftime('%H:%M:%S')}] {msg}"
    print(linha)
    with open(LOG, "a", encoding="utf-8") as f:
        f.write(linha + "\n")

prioridade = sys.argv[1]
nome = sys.argv[2]
comando = sys.argv[3:]

os.makedirs(SPOOL, exist_ok=True)
ticket = f"{prioridade}_{time.time_ns()}_{nome}"
open(os.path.join(SPOOL, ticket), "w", encoding="utf-8").close()
log(f"Job '{nome}' (prio={prioridade}) enfileirado")

# 1) Espera ser o primeiro da fila (ordenacao lexicografica dos tickets)
while True:
    fila = sorted(os.listdir(SPOOL))
    if fila and fila[0] == ticket:
        break
    time.sleep(0.2)

# 2) Adquire o lock (mkdir e atomico: so um processo consegue)
while True:
    try:
        os.mkdir(LOCK)
        break
    except FileExistsError:
        time.sleep(0.2)

# 3) Executa com exclusividade e libera no fim (mesmo se der erro)
try:
    log(f"Executando '{nome}' na GPU")
    subprocess.run([sys.executable] + comando)
    log(f"'{nome}' concluido")
finally:
    try:
        os.rmdir(LOCK)
    except OSError:
        pass
    caminho = os.path.join(SPOOL, ticket)
    if os.path.exists(caminho):
        os.remove(caminho)


In [ ]:
# @title ▶️ Pratica 1: lancar 4 jobs concorrentes e ver a serializacao
# ============================================================================
# OBJETIVO: subir 4 processos ao mesmo tempo e observar que o lock garante
# UM job por vez, na ordem de prioridade (1 antes de 2 antes de 3).
# ============================================================================
import os
import shutil
import subprocess
import sys

# Limpa estado de execucoes anteriores
shutil.rmtree("gpu_spool", ignore_errors=True)
shutil.rmtree("gpu.lockdir", ignore_errors=True)
if os.path.exists("fila.log"):
    os.remove("fila.log")

jobs = [
    ("1", "Job-Alta-A",  ["train_job.py", "--nome", "Job-Alta-A",  "--epocas", "2"]),
    ("3", "Job-Baixa-B", ["train_job.py", "--nome", "Job-Baixa-B", "--epocas", "2"]),
    ("2", "Job-Media-C", ["train_job.py", "--nome", "Job-Media-C", "--epocas", "2"]),
    ("1", "Job-Alta-D",  ["train_job.py", "--nome", "Job-Alta-D",  "--epocas", "2"]),
]

print("Lancando 4 jobs concorrentes...")
procs = [subprocess.Popen([sys.executable, "fila_gpu.py", p, n] + cmd) for p, n, cmd in jobs]
for pr in procs:
    pr.wait()

print("\n=== Log da fila (ordem real de execucao) ===")
print(open("fila.log", encoding="utf-8").read())


## Parte 2 — Energia

### 4. TDP, throttling e o custo escondido

**TDP/TGP** é o teto de consumo da placa. Acima de ~83–87 °C a GPU faz **thermal throttling**
(reduz o clock sozinha): perde desempenho e gasta mais. GPUs no TDP máximo em baixa carga são a
causa clássica de contas de luz acima do projetado.


In [ ]:
# @title 🔌 Leitura de temperatura/potencia (real via pynvml/nvidia-smi)
# ============================================================================
# OBJETIVO: ler metricas reais da GPU. Sem GPU NVIDIA, gera valores plausiveis
# (modo simulado) para que a aula continue rodando.
# ============================================================================
import random

_NVML = None  # cache do (modulo, handle) para nao reinicializar a cada leitura

def _handle_nvml():
    global _NVML
    if _NVML is None:
        import pynvml
        pynvml.nvmlInit()
        _NVML = (pynvml, pynvml.nvmlDeviceGetHandleByIndex(0))
    return _NVML

def ler_gpu():
    """Le temperatura, potencia, limite, uso e clock. Fonte real ou simulada."""
    # 1) pynvml (mais completo)
    if TEM_NVML:
        try:
            pynvml, h = _handle_nvml()
            return {
                "fonte": "pynvml (real)",
                "temp_c": float(pynvml.nvmlDeviceGetTemperature(h, pynvml.NVML_TEMPERATURE_GPU)),
                "power_w": pynvml.nvmlDeviceGetPowerUsage(h) / 1000.0,
                "power_limit_w": pynvml.nvmlDeviceGetPowerManagementLimit(h) / 1000.0,
                "util_pct": float(pynvml.nvmlDeviceGetUtilizationRates(h).gpu),
            }
        except Exception:
            pass
    # 2) nvidia-smi (texto)
    if TEM_SMI:
        try:
            out = subprocess.check_output(
                ["nvidia-smi",
                 "--query-gpu=temperature.gpu,power.draw,enforced.power.limit,utilization.gpu",
                 "--format=csv,noheader,nounits"], text=True).strip().split("\n")[0]
            t, p, pl, u = [float(x) for x in out.split(",")]
            return {"fonte": "nvidia-smi (real)", "temp_c": t, "power_w": p,
                    "power_limit_w": pl, "util_pct": u}
        except Exception:
            pass
    # 3) simulado (sem GPU)
    u = random.uniform(60, 95)
    return {"fonte": "simulado", "temp_c": round(50 + u * 0.35, 1),
            "power_w": round(40 + u * 0.7, 1), "power_limit_w": 70.0,
            "util_pct": round(u, 1)}

amostra = ler_gpu()
print("Amostra atual da GPU:")
for k, v in amostra.items():
    print(f"  {k:<14}: {v}")


In [ ]:
# @title 🌡️ Monitoramento continuo (CSV real + grafico)
# ============================================================================
# OBJETIVO: coletar varias amostras, salvar em CSV (como no laboratorio) e
# plotar a serie temporal. Com GPU real, e exatamente a telemetria do lab.
# ============================================================================
import csv
import time
import matplotlib.pyplot as plt

ARQUIVO = "gpu_thermal.csv"
N_AMOSTRAS = 8
INTERVALO = 0.5

registros = []
print(f"Coletando {N_AMOSTRAS} amostras (a cada {INTERVALO}s)...")
for i in range(N_AMOSTRAS):
    d = ler_gpu()
    d["t"] = round(i * INTERVALO, 1)
    registros.append(d)
    print(f"  t={d['t']:>4}s | {d['fonte']:<16} | {d['temp_c']:>5}C | "
          f"{d['power_w']:>5}W/{d['power_limit_w']:.0f}W | util {d['util_pct']:.0f}%")
    time.sleep(INTERVALO)

with open(ARQUIVO, "w", newline="", encoding="utf-8") as f:
    campos = ["t", "fonte", "temp_c", "power_w", "power_limit_w", "util_pct"]
    w = csv.DictWriter(f, fieldnames=campos)
    w.writeheader()
    w.writerows(registros)

ts = [r["t"] for r in registros]
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(ts, [r["temp_c"] for r in registros], "o-", color="#F97316")
ax[0].axhline(83, color="red", ls=":", label="limite de throttling (83C)")
ax[0].set_title("Temperatura"); ax[0].set_xlabel("tempo (s)"); ax[0].set_ylabel("C"); ax[0].legend()
ax[1].plot(ts, [r["power_w"] for r in registros], "s-", color="#6366F1", label="potencia (W)")
ax[1].plot(ts, [r["power_limit_w"] for r in registros], "--", color="#9aa7d4", label="limite (W)")
ax[1].set_title("Potencia"); ax[1].set_xlabel("tempo (s)"); ax[1].set_ylabel("W"); ax[1].legend()
plt.tight_layout(); plt.show()
print(f"Historico salvo em {ARQUIVO}  (fonte: {registros[0]['fonte']})")


### 5. Throughput e eficiência (imgs/J)

Agora medimos o **throughput real** de um treino curto (imagens/segundo) e a **potência média**
durante ele — daí sai a **eficiência (imgs/J)**. Esse é o número que conecta desempenho e
consumo.


In [ ]:
# @title ⚡ Medir throughput real + potencia media -> eficiencia (imgs/J)
# ============================================================================
# OBJETIVO: rodar um treino curto e medir quantas imagens por segundo a GPU
# processa e quanta potencia consome em media. Eficiencia = imgs/s / W.
# ============================================================================
import time

def medir_treino(segundos=6, batch=32):
    """Treina por 'segundos' e devolve throughput e potencia media."""
    import torch
    import torch.nn as nn
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    modelo = nn.Sequential(
        nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(),
        nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(16, 10),
    ).to(dev)
    otimizador = torch.optim.SGD(modelo.parameters(), lr=0.01)
    criterio = nn.CrossEntropyLoss()
    modelo(torch.randn(batch, 3, 64, 64, device=dev))  # warm-up
    if dev == "cuda":
        torch.cuda.synchronize()
    t0 = time.time()
    imagens = 0
    potencias = []
    while time.time() - t0 < segundos:
        x = torch.randn(batch, 3, 64, 64, device=dev)
        y = torch.randint(0, 10, (batch,), device=dev)
        otimizador.zero_grad()
        perda = criterio(modelo(x), y)
        perda.backward()
        otimizador.step()
        if dev == "cuda":
            torch.cuda.synchronize()
        imagens += batch
        potencias.append(ler_gpu()["power_w"])
    duracao = time.time() - t0
    return {"device": dev, "imgs_s": imagens / duracao,
            "power_w": sum(potencias) / len(potencias)}

if TEM_TORCH:
    res = medir_treino()
    efic = res["imgs_s"] / res["power_w"]
    print(f"Dispositivo     : {res['device'].upper()}")
    print(f"Throughput      : {res['imgs_s']:.1f} imgs/s")
    print(f"Potencia media  : {res['power_w']:.1f} W")
    print(f"Eficiencia      : {efic:.3f} imgs/J")
    if res["device"] == "cpu":
        print("\n(Em CPU a potencia nao e da GPU; rode no Colab com T4 para a medicao real.)")
else:
    print("PyTorch nao instalado - sem medicao de throughput.")


### 6. Power Limit: o controle mais eficaz

Reduzir o **Power Limit** em 20–30% costuma custar só 5–10% de throughput. Como configurar o PL
exige `sudo` (indisponível no Colab), usamos o **modelo logístico** — o mesmo do laboratório —
para mostrar o trade-off e localizar o ponto de melhor **eficiência**.


In [ ]:
# @title ⚙️ Trade-off do Power Limit: throughput x eficiencia
# ============================================================================
# OBJETIVO: estimar o PL de melhor eficiencia (imgs/J) com o modelo logistico.
# No laboratorio Linux, o mesmo grafico usa PLs aplicados de verdade via
# `nvidia-smi -pl`.
# ============================================================================
import math

TDP = 250  # potencia maxima de referencia (W)

def throughput_modelo(pl, tdp=TDP):
    x = pl / tdp
    return 480.0 / (1.0 + math.exp(-6.0 * (x - 0.5)))

resultados = []
print(f"{'PL (W)':>8} {'Imgs/s':>10} {'Pwr (W)':>10} {'Imgs/J':>10}")
print("-" * 42)
for pl in [100, 150, 200, 225, 250]:
    imgs = throughput_modelo(pl)
    pwr = pl * 0.92
    efic = imgs / pwr
    resultados.append((pl, imgs, pwr, efic))
    print(f"{pl:>8} {imgs:>10.1f} {pwr:>10.1f} {efic:>10.3f}")

melhor = max(resultados, key=lambda r: r[3])
print(f"\nPonto otimo: PL={melhor[0]}W (~{melhor[0]/TDP:.0%} do TDP), {melhor[3]:.3f} imgs/J")


In [ ]:
# @title 📊 Grafico do trade-off (throughput x eficiencia)
# ============================================================================
# OBJETIVO: visualizar por que o melhor PL nao e o de potencia maxima.
# ============================================================================
import matplotlib.pyplot as plt

pls = [r[0] for r in resultados]
imgs = [r[1] for r in resultados]
efics = [r[3] for r in resultados]

fig, ax1 = plt.subplots(figsize=(9, 5))
ax1.plot(pls, imgs, "o-", color="#6366F1", label="Throughput (imgs/s)")
ax1.set_xlabel("Power Limit (W)")
ax1.set_ylabel("Throughput (imgs/s)", color="#6366F1")
ax1.tick_params(axis="y", labelcolor="#6366F1")
ax2 = ax1.twinx()
ax2.plot(pls, efics, "s--", color="#10B981", label="Eficiencia (imgs/J)")
ax2.set_ylabel("Eficiencia (imgs/J)", color="#10B981")
ax2.tick_params(axis="y", labelcolor="#10B981")
ax1.axvline(x=melhor[0], color="#F97316", linestyle=":", alpha=0.7)
ax1.set_title("Trade-off: Desempenho x Eficiencia Energetica")
plt.tight_layout()
plt.savefig("tradeoff_energia.png", dpi=120)
plt.show()
print(f"Melhor eficiencia em PL={melhor[0]}W.")


In [ ]:
# @title 🔧 Faixa de Power Limit suportada pela GPU (NVML real)
# ============================================================================
# OBJETIVO: mostrar quanto a placa aceita de ajuste de PL. Requer GPU NVIDIA.
# ============================================================================
try:
    pynvml, h = _handle_nvml()
    pl_min, pl_max = pynvml.nvmlDeviceGetPowerManagementLimitConstraints(h)
    pl_atual = pynvml.nvmlDeviceGetPowerManagementLimit(h) / 1000.0
    print(f"Power Limit atual : {pl_atual:.0f} W")
    print(f"Faixa suportada   : {pl_min/1000:.0f} W - {pl_max/1000:.0f} W")
    print("Ajuste real: sudo nvidia-smi -pl <watts> (fora do Colab).")
except Exception as e:
    print("[SIMULADO] NVML indisponivel:", e)
    print("Faixa de referencia (Tesla T4): 60 W - 70 W")


## Integração — juntando as duas partes

Só faz sentido medir energia **com a GPU exclusiva**: dois jobs juntos contaminam os dados. Por
isso o **lock vem primeiro**. Abaixo, medimos a eficiência de um job que já detém o lock.


In [ ]:
# @title 🧩 Job energetico DENTRO do lock (integra concorrencia + energia)
# ============================================================================
# OBJETIVO: adquirir o lock e, com a GPU exclusiva, reportar a eficiencia.
# ============================================================================
import os
import shutil
import time

LOCK = "gpu.lockdir"
shutil.rmtree(LOCK, ignore_errors=True)

def tentar_lock():
    try:
        os.mkdir(LOCK)
        return True
    except FileExistsError:
        return False

if tentar_lock():
    print("Lock adquirido - medindo eficiencia com a GPU exclusiva...")
    if TEM_TORCH:
        r = medir_treino(segundos=4)
        print(f"Dispositivo={r['device'].upper()} | {r['imgs_s']:.1f} imgs/s | "
              f"{r['power_w']:.1f} W | {r['imgs_s']/r['power_w']:.3f} imgs/J")
    else:
        print("Sem PyTorch - medicao indisponivel.")
    shutil.rmtree(LOCK, ignore_errors=True)
else:
    print("GPU ocupada por outro job - tente novamente.")


## 📝 Exercícios (5)

Resolva os 5 exercícios **neste notebook**. O valor está em **experimentar e explicar**.

---

**1) Sem lock.** Rode a célula da race condition com VRAM de 8 GB (em vez de 4). Quantos jobs
cabem? A OOM ainda acontece? O que isso ensina sobre planejar a VRAM?

**2) Fila com aging.** Um job de prioridade 3 pode nunca rodar (starvation). Descreva, em
pseudocódigo, como o **aging** (envelhecer o ticket) resolve isso.

**3) Medição real.** Com GPU (Colab T4), anote a **temperatura**, a **potência** e o
**throughput** medidos. Sem GPU, use os valores de referência e diga o que mudaria.

**4) Eficiência (imgs/J).** Calcule a eficiência do seu treino (`imgs/s ÷ W`). No modelo do
Power Limit, qual PL deu a **melhor eficiência**? O ganho de 200 W→250 W compensa o consumo?

**5) Integração.** Escreva um plano (5–8 linhas) para operar **1 GPU com 3 usuários**,
combinando o **lock/fila** e o **Power Limit** desta aula.

---

### Célula-esqueleto (complete para o exercício 2)

```python
# TODO: envelhecer tickets antigos (aging)
# import os, time
# SPOOL = "gpu_spool"
# for ticket, ...:
#     idade = time.time() - os.path.getmtime(...)
#     if idade > 10:  # rebaixa a prioridade do ticket
#         ...
```


---

## 📝 Questionário de Consolidação (Aulas 14 a 19)

As **18 questões** de revisão do Bloco 3 estão na pasta de questionários do curso:

- **Questionário:** [`questionarios/questionario-aulas-14-19.md`](../../../../questionarios/questionario-aulas-14-19.md)

> **Entrega:** envie as respostas por e-mail para `03049691093@senacrs.edu.br` com o assunto
> `Questionario aulas 14 a 19`.
